# Lab 03 · OpenMP · your first parallel program

Lab 01 gave you serial `heat2D.c`. Lab 02 measured it and put it on a roofline. Now you parallelize with **OpenMP** — the shared-memory model that lets one process use many cores of the same node — and measure whether the speedup matches what the roofline predicted.

**Prerequisites.** Lab 01 (working serial `heat2D.c`), Lab 02 (roofline for Crux). Skim labCC Part 6 (reduction-order reproducibility) — you'll hit it when you sum across threads.

**Builds toward.** **Lab 04** covers OpenMP pitfalls (races, false sharing, scheduling). **Lab 07** combines OpenMP with MPI. **Lab 08** takes the same OpenMP program and offloads it to a GPU.

> **📚 Where to look when you're stuck**
>
> - [**OpenMP 5.2 spec (short)**](https://www.openmp.org/spec-html/5.2/openmp.html)
> - [**LLNL OpenMP tutorial**](https://hpc-tutorials.llnl.gov/openmp/)
> - [**Crux compute nodes**](https://docs.alcf.anl.gov/crux/) — AMD EPYC, 128 cores/node; you'll sweep 1→128.



## How this notebook works

Same three surfaces as lab 01 and 02: **[Hub]** for orchestration, **[Hub -> Crux]** for ssh calls, and **[Crux compute]** inside the PBS script. Every code cell begins with `# [Where]`.


In [ ]:
# [Hub] Shared toolkit.
from labHelpers import *


### Set up this lab's identity


In [ ]:
# [Hub] Change HPC_USER; re-run.
env = setupLab(labName="lab03", host="crux",
               remoteUser=os.environ.get("HPC_USER","CHANGE_ME"),
               project="UIC-CS455-Sp2027", queue="debug",
               scratch=f"/eagle/UIC-CS455-Sp2027/{os.environ.get('HPC_USER','CHANGE_ME')}")
labDir = pathlib.Path(env['labDir'])


### Preflight


In [ ]:
# [Hub] Reachability + previous lab's artifact.
preflight([
    check("passwordless ssh", sshReachable()),
    check("scheduler answers", schedulerAnswers()),
    check("lab03 dir on cluster", remoteFileExists(env['HPC_LAB_DIR']),
          hint="next cell creates it if missing"),
    check("lab01 heat2D binary", remoteFileExists(env['HPC_LAB_DIR'].replace('lab03','lab01') + '/heat2D')),
], infoRows=[('cluster', clusterHost()), ('you', env.get('HPC_USER','?')),
             ('project', env.get('HPC_PROJECT','?')),
             ('lab dir', env.get('HPC_LAB_DIR','?'))])


In [ ]:
# [Hub -> Crux] Make the lab dir if missing.
sshRun(f'mkdir -p {env["HPC_LAB_DIR"]}/out', quiet=True)
print('lab03 dir ready')


## Part 1 · The `#pragma omp parallel for`

OpenMP is a set of compiler directives (`#pragma omp …`) plus a small runtime library. The compiler flag `-fopenmp` turns the pragmas on; without it they're silently ignored (a common newbie confusion).

The simplest transformation for a stencil is putting `#pragma omp parallel for` on the outer loop of the update. Every iteration of the outer loop becomes a task for one thread. Because the inner loop reads only `u[i-1..i+1][j-1..j+1]` and writes only `unew[i][j]`, there are no data races between threads.

See [LLNL OpenMP tutorial](https://hpc-tutorials.llnl.gov/openmp/) for the authoritative walkthrough.


In [ ]:
# [Hub -> Crux] Fetch lab01's heat2D.c, add one pragma, ship it back.
sshGet(env['HPC_LAB_DIR'].replace('lab03','lab01') + '/heat2D.c',
       str(labDir/'heat2Dserial.c'))
src = (labDir/'heat2Dserial.c').read_text()
# Insert #include and the pragma above the outer i-loop of the update.
if '#include <omp.h>' not in src:
    src = src.replace('#include <stdio.h>', '#include <stdio.h>\n#include <omp.h>', 1)
if '#pragma omp parallel for' not in src:
    src = re.sub(r'(for\s*\(\s*int\s+i\s*=\s*1\s*;)',
                 r'#pragma omp parallel for schedule(static)\n    \1', src, count=1)
(labDir/'heat2Domp.c').write_text(src)
showFile(labDir/'heat2Domp.c', language='c', maxLines=40, title='heat2Domp.c (top)')


In [ ]:
checkpoint("Part 1 - parallelized source", [
    check("heat2Domp.c has omp pragma",
          fileContains(str(labDir/'heat2Domp.c'), '#pragma omp parallel for')),
    check("heat2Domp.c includes omp.h",
          fileContains(str(labDir/'heat2Domp.c'), 'omp.h')),
])


## Part 2 · Sweep thread counts, extend `timings.csv`

Build with `-fopenmp`, sweep `OMP_NUM_THREADS` across 1, 2, 4, 8, 16, 32, 64, 128 on one Crux compute node, and append rows to your `timings.csv` with `variant='openmp'`.

The AMD EPYC node in Crux has 128 physical cores organized as two 64-core sockets. You'll cross a NUMA boundary somewhere in the sweep and probably see a discontinuity — that's lab 04's problem.


In [ ]:
# [Hub -> Crux] Ship, build, sweep thread counts, collect CSV rows.
sshPut(str(labDir/'heat2Domp.c'), env['HPC_LAB_DIR']+'/heat2Domp.c')
jobBody = f'''cd {env["HPC_LAB_DIR"]}
cc -O3 -fopenmp -Wall -o heat2Domp heat2Domp.c -lm
for t in 1 2 4 8 16 32 64 128; do
  OMP_NUM_THREADS=$t OMP_PROC_BIND=close OMP_PLACES=cores \\
    ./heat2Domp --N 1024 --steps 500 --snapEvery 0 --outDir ./out --variant openmp
done
cat ./out/timings.csv
'''
pbsPath = labDir/'sweepJob.pbs'
pbsPath.write_text(pbsHeader(name='lab03Sweep', project=env['HPC_PROJECT'],
                             queue=env.get('HPC_QUEUE','debug'), walltime='00:30:00',
                             filesystems='home:eagle',
                             outPath=env['HPC_LAB_DIR']+'/sweep.out') + jobBody)
sshPut(str(pbsPath), env['HPC_LAB_DIR']+'/sweepJob.pbs')
jobID = submitJob(env['HPC_LAB_DIR']+'/sweepJob.pbs')
print('submitted:', jobID); waitJob(jobID, 30, 2400)
sshGet(env['HPC_LAB_DIR']+'/out/timings.csv', str(labDir/'timings.csv'))
sshGet(env['HPC_LAB_DIR']+'/sweep.out', str(labDir/'sweep.out'))
print((labDir/'sweep.out').read_text()[-800:])


In [ ]:
checkpoint("Part 2 - thread sweep", [
    check("sweep completed", fileExists(str(labDir/'sweep.out'))),
    check("timings.csv has 8 openmp rows",
          lambda: (sum(1 for _ in open(labDir/'timings.csv'))-1 >= 8, 'ok')),
])


## Part 3 · Plot speedup and efficiency

You know this shape from labDD. Now with real data.


In [ ]:
# [Hub] Speedup + efficiency plot for the OpenMP sweep.
import pandas as pd
df = pd.read_csv(labDir/'timings.csv')
omp = df[df['variant']=='openmp'].sort_values('threads').reset_index(drop=True)
print(omp[['threads','wall_s','mlups']].to_string(index=False))
r = plotScaling(str(labDir/'timings.csv'), kind='strong', variantFilter='openmp',
                baselineCol='threads', timeCol='wall_s',
                outPath=str(labDir/'figures'/'ompStrong'))
print('wrote:', [str(p) for p in r])


In [ ]:
checkpoint("Part 3 - scaling plot", [
    check("strong scaling PDF written",
          fileExists(str(labDir/'figures'/'ompStrong.pdf'))),
])


## Part 4 · Compare to the roofline prediction

Lab 02 built a roofline for Crux and showed the serial stencil sits well below the memory-bandwidth ceiling. If the stencil is memory-bound, more threads on the same node buy performance **only** by keeping the memory controllers busier. Once you saturate DRAM bandwidth, adding threads stops helping.

Look at your Part 3 plot: at what thread count does speedup flatten? That's the point at which you've saturated one socket's memory controllers.


In [ ]:
# [Hub] Compute the plateau: where does mlups stop climbing?
omp = pd.read_csv(labDir/'timings.csv').query("variant=='openmp'").sort_values('threads')
peakMlups = omp['mlups'].max()
plateauRows = omp[omp['mlups'] >= 0.9 * peakMlups]
print(f'peak throughput: {peakMlups:.1f} MLUP/s')
print(f'first thread count within 10% of peak: {plateauRows["threads"].min()}')
print(f'threads beyond which no gain: {plateauRows["threads"].min()}')


In [ ]:
checkpoint("Part 4 - plateau identified", [
    check("peak mlups computed", lambda: (True, f'{peakMlups:.1f} MLUP/s')),
])


## Part 5 · Correctness · does the parallel version give the same answer?

Every parallel version you write from here on gets validated against the serial reference. For a diffusion problem, we check **sum-of-u conservation** (the field integral shouldn't drift) and compare the final `snapshot_*.npy` frames against lab 01's.


In [ ]:
# [Hub -> Crux] Run the serial + omp versions with identical params, diff.
jobBody = f'''cd {env["HPC_LAB_DIR"]}
cp ../lab01/heat2D .
./heat2D    --N 256 --steps 200 --snapEvery 200 --outDir ./chkSerial
OMP_NUM_THREADS=8 ./heat2Domp --N 256 --steps 200 --snapEvery 200 --outDir ./chkOMP
'''
pbsPath = labDir/'chkJob.pbs'
pbsPath.write_text(pbsHeader(name='lab03Chk', project=env['HPC_PROJECT'],
                             queue=env.get('HPC_QUEUE','debug'), walltime='00:10:00',
                             filesystems='home:eagle',
                             outPath=env['HPC_LAB_DIR']+'/chk.out') + jobBody)
sshPut(str(pbsPath), env['HPC_LAB_DIR']+'/chkJob.pbs')
jobID = submitJob(env['HPC_LAB_DIR']+'/chkJob.pbs'); waitJob(jobID, 15, 900)
sshGet(env['HPC_LAB_DIR']+'/chkSerial/snapshot_00200.npy', str(labDir/'refSerial.npy'))
sshGet(env['HPC_LAB_DIR']+'/chkOMP/snapshot_00200.npy',    str(labDir/'refOMP.npy'))
import numpy as np
a = np.load(labDir/'refSerial.npy'); b = np.load(labDir/'refOMP.npy')
diff = float(np.abs(a-b).max())
rel  = diff / max(abs(a).max(), 1e-30)
print(f'max abs diff: {diff:.3e}   relative: {rel:.3e}')
showNote('OpenMP matches serial to floating-point noise (labCC Part 6).' if rel < 1e-10
         else f'OpenMP DIFFERS from serial (rel err {rel:.2e}). Investigate before proceeding.',
         kind='ok' if rel < 1e-10 else 'warn')


In [ ]:
checkpoint("Part 5 - correctness", [
    check("serial reference snapshot",  fileExists(str(labDir/'refSerial.npy'))),
    check("OpenMP snapshot",           fileExists(str(labDir/'refOMP.npy'))),
])


## Part 6 · Bridge to lab 04

OpenMP with one pragma works. Now the fun part — everything that can go wrong.

**Lab 04 covers:**
- **Data races** when the pragma is wrong (`#pragma omp parallel for` on the wrong loop)
- **False sharing** — two threads writing near each other in memory, cache ping-ponging kills performance
- **NUMA effects** — first-touch matters when you cross a socket boundary
- **Scheduling** — `static`, `dynamic`, `guided`, and when each is right

Every one of these will show up as a mystery slowdown or wrong answer on the sweep you just ran. Lab 04 is where you learn to spot them.


## Wrap up

Moved the spine forward one lab. Ready for the next.


### Lab scorecard


In [ ]:
labSummary("OpenMP")


---
### One-minute feedback

What worked, what didn't, what should be clearer.


In [ ]:
feedback("OpenMP")
